# Módulo 15: Soluciones basadas en datos — Guía del ejercicio

En este ejercicio vas a construir el **lienzo de modelo de negocio** (*Business Model Canvas*) de una cadena de gimnasios, a señalar en qué bloques la ciencia de datos crea valor y a comprobarlo con datos: segmentarás a sus clientes, medirás el abandono de cada segmento y estimarás cuánto vale cada cliente a lo largo de su vida.

- Si necesitas repasar los conceptos, consulta la lección [m15-soluciones-basadas-en-datos-leccion.md](m15-soluciones-basadas-en-datos-leccion.md).
- Cuando termines, compara tu trabajo con la solución: [m15-soluciones-basadas-en-datos-solucion.ipynb](m15-soluciones-basadas-en-datos-solucion.ipynb).

## Contexto

**PulsoFit** es una cadena de 14 gimnasios en ciudades medianas. Vende membresías mensuales sin permanencia en tres planes: **Básico** (29 $ al mes, sala de máquinas), **Plus** (45 $, añade clases grupales) y **Premium** (69 $, añade todas las sedes, piscina y una sesión mensual con entrenador). Además cobra extras: clases especiales, entrenamiento personal y productos de su tienda.

El negocio crece en altas, pero la dirección está preocupada: muchos clientes cancelan a los pocos meses y cada nuevo cliente cuesta caro en publicidad y promociones de inscripción. La directora general quiere entender qué papel puede jugar la ciencia de datos en el modelo de negocio y, en concreto, **qué clientes se van, cuánto vale cada tipo de cliente y dónde conviene invertir para retenerlos**.

Para el cálculo del valor de vida del cliente, el área financiera estima que el **margen de contribución es del 40 %** de lo que paga cada cliente: el resto cubre instalaciones, personal y energía.

## Datos

La celda inicial genera el DataFrame `clientes`, una muestra de 2 400 clientes activos en la fecha de corte. Tiene una fila por cliente y estas columnas:

| Columna | Descripción |
| --- | --- |
| `cliente_id` | Identificador del cliente |
| `plan` | Plan contratado: `Básico`, `Plus` o `Premium` |
| `antiguedad_meses` | Meses como cliente en la fecha de corte |
| `visitas_mes` | Visitas promedio al mes en los tres meses previos al corte |
| `gasto_mensual` | Gasto promedio mensual en $ (cuota del plan más extras) |
| `abandono` | 1 si el cliente canceló su membresía en los **3 meses siguientes** al corte; 0 si siguió activo |

Los datos son sintéticos y se generan con una semilla fija, así que siempre obtendrás los mismos valores.

## Instrucciones

- Trabaja en este notebook (`uv run jupyter lab` desde la raíz del repositorio) y ejecuta primero la celda inicial.
- Las tareas 1 a 3 son preguntas escritas sobre el caso: responde en la celda de texto que sigue a cada una. Para el lienzo puedes usar una tabla de Markdown, como en la lección.
- Las tareas 4 a 7 son de Python: escribe tu código en las celdas vacías y añade más celdas si lo necesitas.
- Acompaña cada resultado numérico con una interpretación breve en términos del negocio.

La celda siguiente importa las librerías y genera los datos.

In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "1"  # evita un aviso de KMeans en Windows

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 2400

# Perfil latente de cada cliente (no forma parte de los datos que analizas)
perfil = rng.choice(4, size=n, p=[0.30, 0.35, 0.20, 0.15])
antiguedad = np.select(
    [perfil == 0, perfil == 1, perfil == 2, perfil == 3],
    [rng.integers(1, 7, n), rng.integers(6, 37, n), rng.integers(18, 85, n), rng.integers(4, 31, n)],
)
visitas = np.select(
    [perfil == 0, perfil == 1, perfil == 2, perfil == 3],
    [rng.gamma(4, 0.8, n), rng.normal(8, 1.8, n), rng.normal(14, 2.5, n), rng.gamma(4, 0.8, n)],
)
visitas = np.clip(visitas, 0.3, 24).round(1)

prob_plan = {0: [0.75, 0.20, 0.05], 1: [0.50, 0.40, 0.10], 2: [0.15, 0.40, 0.45], 3: [0.05, 0.25, 0.70]}
plan = np.array([rng.choice(["Básico", "Plus", "Premium"], p=prob_plan[k]) for k in perfil])
cuota = pd.Series(plan).map({"Básico": 29, "Plus": 45, "Premium": 69}).to_numpy()
extras = rng.gamma(2, np.select([perfil == 0, perfil == 1, perfil == 2, perfil == 3], [3, 6, 14, 18]))
gasto = (cuota + extras).round(2)

# Probabilidad de abandono en los 3 meses siguientes
logit = 0.5 - 0.15 * visitas - 0.25 * np.log(antiguedad)
abandono = rng.binomial(1, 1 / (1 + np.exp(-logit)))

clientes = pd.DataFrame({
    "cliente_id": np.arange(1, n + 1),
    "plan": plan,
    "antiguedad_meses": antiguedad,
    "visitas_mes": visitas,
    "gasto_mensual": gasto,
    "abandono": abandono,
})
clientes.head()

## Tarea 1. Lienzo de modelo de negocio de PulsoFit

Completa los **nueve bloques** del lienzo de modelo de negocio de PulsoFit, de forma breve y concreta para este caso:

1. Segmentos de clientes.
2. Propuesta de valor.
3. Canales.
4. Relaciones con los clientes.
5. Fuentes de ingresos.
6. Recursos clave.
7. Actividades clave.
8. Socios clave.
9. Estructura de costos.

Usa la información del contexto y supuestos razonables para lo que no se indique.

*Escribe aquí tu respuesta.*

## Tarea 2. Dónde crea valor la ciencia de datos

Elige **al menos cuatro bloques** del lienzo en los que la ciencia de datos cree valor para PulsoFit. Para cada uno indica:

1. Qué decisión o servicio mejora.
2. Qué datos necesitaría (de los que ya genera el gimnasio o de otros nuevos).
3. Qué técnica o tipo de análisis usarías (por ejemplo, segmentación, predicción de abandono, pronóstico de demanda).

Una tabla con las columnas *Bloque*, *Decisión*, *Datos* y *Técnica* es un buen formato.

*Escribe aquí tu respuesta.*

## Tarea 3. Oportunidades y amenazas

1. Propón **tres oportunidades** para PulsoFit en las que los datos sean una ventaja.
2. Señala **tres amenazas** para su modelo de negocio. Incluye al menos una relacionada con los propios datos (privacidad, calidad, sesgos o dependencia de proveedores).
3. Cierra con un balance de dos o tres frases: ¿qué debería priorizar la dirección?

*Escribe aquí tu respuesta.*

## Tarea 4. Segmentar a los clientes

1. Explora `clientes`: dimensiones, estadísticos descriptivos de las columnas numéricas y número de clientes por plan.
2. Estandariza `antiguedad_meses`, `visitas_mes` y `gasto_mensual` con `StandardScaler` de scikit-learn. Explica por qué hace falta estandarizar antes de KMeans.
3. Aplica `KMeans` con `n_clusters=4`, `n_init=10` y `random_state=42` y guarda la etiqueta de cada cliente en una columna `segmento`.
4. Describe cada segmento con su tamaño y la mediana de antigüedad, visitas y gasto. Ponle a cada uno un **nombre de negocio** que lo describa (por ejemplo, "nuevos ocasionales"). Ten en cuenta que los números de clúster que asigna KMeans son arbitrarios.
5. Muestra en un gráfico de dispersión las visitas frente al gasto, con un color por segmento.

In [ ]:
# Tu código aquí

*Escribe aquí tu interpretación.*

## Tarea 5. Tasa de abandono por segmento

1. Calcula la tasa de abandono a 3 meses de cada segmento (la media de `abandono`) y el número de clientes que abandonaron.
2. Calcula también la tasa de abandono por `plan`.
3. Representa la tasa por segmento en un gráfico de barras ordenado de mayor a menor.
4. ¿Qué segmento pierde más clientes? ¿Explica el plan contratado el abandono tan bien como el comportamiento (visitas y antigüedad)?

In [ ]:
# Tu código aquí

*Escribe aquí tu interpretación.*

## Tarea 6. Valor de vida del cliente (CLV)

Estima el valor de vida del cliente (*Customer Lifetime Value*, CLV) de cada segmento con esta fórmula simple, sin descuento financiero:

$$\text{CLV} = \text{gasto mensual medio} \times \text{margen} \times \text{vida esperada en meses}$$

1. Convierte la tasa de abandono a 3 meses, $p_3$, en una **tasa mensual** suponiendo que cada mes se pierde la misma proporción de clientes: $c = 1 - (1 - p_3)^{1/3}$.
2. Calcula la vida esperada como $1 / c$ meses.
3. Calcula el CLV por segmento con un margen de 0.40 y presenta una tabla con gasto medio, $p_3$, $c$, vida esperada y CLV, redondeada.
4. Explica qué supuestos hay detrás de esta fórmula y en qué segmento retener a un cliente vale más.

In [ ]:
# Tu código aquí

*Escribe aquí tu interpretación.*

## Tarea 7. Del análisis al lienzo

1. Construye una tabla por segmento con: porcentaje de clientes, porcentaje del ingreso mensual total y **ingreso mensual en riesgo** (la suma de `gasto_mensual` de los clientes que abandonaron).
2. Con esa tabla y los resultados de las tareas 4 a 6, explica cómo cambiarías tres bloques del lienzo de la Tarea 1: **segmentos de clientes**, **propuesta de valor** y **fuentes de ingresos**. Propón al menos una acción concreta para el segmento más valioso en riesgo.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Antes de terminar

- [ ] El lienzo tiene los nueve bloques, incluido el de segmentos de clientes.
- [ ] Señalaste al menos cuatro bloques donde la ciencia de datos crea valor, con datos y técnica.
- [ ] Las variables se estandarizaron antes de KMeans y cada segmento tiene un nombre de negocio.
- [ ] El CLV usa la tasa de abandono **mensual**, no la de 3 meses.
- [ ] Las conclusiones conectan los números con los bloques del lienzo.

Cuando termines, compara tu trabajo con la solución: [m15-soluciones-basadas-en-datos-solucion.ipynb](m15-soluciones-basadas-en-datos-solucion.ipynb).